# Telco Customer Churn Prediction

**Dataset:** IBM Telco Customer Churn  
**Objective:** Predict whether a customer will churn using machine learning models.  
**Models Used:** Logistic Regression, Random Forest  

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 100
print('Libraries loaded successfully.')

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Telco-Customer-Churn.csv')
print('Dataset Shape:', df.shape)
df.head()

In [ ]:
print('Column Names:', df.columns.tolist())
print('\nData Types:')
print(df.dtypes)

## 3. Data Cleaning

In [ ]:
# Check missing values
print('Missing values per column:')
print(df.isnull().sum())
print('\nTotal missing values:', df.isnull().sum().sum())

In [ ]:
# Fix TotalCharges: stored as string with spaces
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
print('TotalCharges NaN after conversion:', df['TotalCharges'].isnull().sum())

# Drop rows where TotalCharges is NaN (11 rows with 0 tenure)
df = df.dropna(subset=['TotalCharges'])
print('Shape after dropping NaN TotalCharges rows:', df.shape)

In [ ]:
# Check for duplicates
print('Duplicate rows:', df.duplicated().sum())

# Reset index
df = df.reset_index(drop=True)
print('Final dataset shape:', df.shape)

In [ ]:
# Summary statistics
df.describe()

## 4. Exploratory Data Analysis

In [ ]:
# Overall churn distribution
churn_counts = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100
print('Churn Distribution:')
print(pd.DataFrame({'Count': churn_counts, 'Percentage': churn_pct.round(2)}))

fig, ax = plt.subplots(figsize=(5, 4))
colors = ['#3b82d4', '#e05252']
ax.bar(churn_counts.index, churn_counts.values, color=colors)
ax.set_title('Customer Churn Distribution', fontsize=13, fontweight='bold')
ax.set_xlabel('Churn')
ax.set_ylabel('Number of Customers')
for i, v in enumerate(churn_counts.values):
    ax.text(i, v + 30, f'{v}\n({churn_pct.values[i]:.1f}%)', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('churn_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Churn by Contract type
contract_churn = df.groupby('Contract')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).round(2)
print('Churn Rate by Contract Type (%):')
print(contract_churn)

fig, ax = plt.subplots(figsize=(6, 4))
contract_churn.plot(kind='bar', ax=ax, color='#3b82d4', edgecolor='white')
ax.set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
ax.set_xlabel('Contract Type')
ax.set_ylabel('Churn Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
for i, v in enumerate(contract_churn.values):
    ax.text(i, v + 0.5, f'{v}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('churn_by_contract.png', bbox_inches='tight')
plt.show()

In [ ]:
# Churn by Internet Service
internet_churn = df.groupby('InternetService')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).round(2)
print('Churn Rate by Internet Service (%):')
print(internet_churn)

fig, ax = plt.subplots(figsize=(6, 4))
internet_churn.plot(kind='bar', ax=ax, color='#7c5cd8', edgecolor='white')
ax.set_title('Churn Rate by Internet Service', fontsize=13, fontweight='bold')
ax.set_xlabel('Internet Service')
ax.set_ylabel('Churn Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for i, v in enumerate(internet_churn.values):
    ax.text(i, v + 0.5, f'{v}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('churn_by_internet.png', bbox_inches='tight')
plt.show()

In [ ]:
# Churn by Payment Method
payment_churn = df.groupby('PaymentMethod')['Churn'].apply(
    lambda x: (x == 'Yes').mean() * 100
).round(2)
print('Churn Rate by Payment Method (%):')
print(payment_churn)

fig, ax = plt.subplots(figsize=(7, 4))
payment_churn.plot(kind='bar', ax=ax, color='#e05252', edgecolor='white')
ax.set_title('Churn Rate by Payment Method', fontsize=13, fontweight='bold')
ax.set_xlabel('Payment Method')
ax.set_ylabel('Churn Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=20, ha='right')
for i, v in enumerate(payment_churn.values):
    ax.text(i, v + 0.5, f'{v}%', ha='center', fontsize=10)
plt.tight_layout()
plt.savefig('churn_by_payment.png', bbox_inches='tight')
plt.show()

In [ ]:
# Tenure distribution by Churn
print('Average Tenure:')
print('  Churned:', round(df[df['Churn'] == 'Yes']['tenure'].mean(), 2), 'months')
print('  Not Churned:', round(df[df['Churn'] == 'No']['tenure'].mean(), 2), 'months')

fig, ax = plt.subplots(figsize=(7, 4))
df[df['Churn'] == 'No']['tenure'].plot(kind='hist', bins=30, ax=ax,
    alpha=0.7, color='#3b82d4', label='Not Churned')
df[df['Churn'] == 'Yes']['tenure'].plot(kind='hist', bins=30, ax=ax,
    alpha=0.7, color='#e05252', label='Churned')
ax.set_title('Tenure Distribution by Churn', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure (months)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.savefig('tenure_distribution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Monthly Charges by Churn
print('Average Monthly Charges:')
print('  Churned: $', round(df[df['Churn'] == 'Yes']['MonthlyCharges'].mean(), 2))
print('  Not Churned: $', round(df[df['Churn'] == 'No']['MonthlyCharges'].mean(), 2))

fig, ax = plt.subplots(figsize=(7, 4))
df.boxplot(column='MonthlyCharges', by='Churn', ax=ax,
           boxprops=dict(color='#3b82d4'),
           medianprops=dict(color='#e05252', linewidth=2))
ax.set_title('Monthly Charges by Churn Status', fontsize=13, fontweight='bold')
plt.suptitle('')
ax.set_xlabel('Churn')
ax.set_ylabel('Monthly Charges ($)')
plt.tight_layout()
plt.savefig('monthly_charges.png', bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap (numerical columns)
df_temp = df.copy()
df_temp['Churn_binary'] = (df_temp['Churn'] == 'Yes').astype(int)
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen', 'Churn_binary']

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(df_temp[num_cols].corr(), annot=True, fmt='.2f',
            cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Correlation Heatmap (Numerical Features)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()

In [ ]:
# Churn rate by tenure group
bins = [0, 12, 24, 36, 48, 60, 72, 999]
labels = ['0-12', '13-24', '25-36', '37-48', '49-60', '61-72', '73+']
df['tenure_group'] = pd.cut(df['tenure'], bins=bins, labels=labels, right=True)
tenure_churn = df.groupby('tenure_group', observed=True)['Churn'].apply(
    lambda x: round((x == 'Yes').mean() * 100, 2)
)
print('Churn Rate by Tenure Group (%):')
print(tenure_churn)

fig, ax = plt.subplots(figsize=(7, 4))
tenure_churn.plot(kind='bar', ax=ax, color='#3b82d4', edgecolor='white')
ax.set_title('Churn Rate by Tenure Group', fontsize=13, fontweight='bold')
ax.set_xlabel('Tenure Group (months)')
ax.set_ylabel('Churn Rate (%)')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
for i, v in enumerate(tenure_churn.values):
    ax.text(i, v + 0.5, f'{v}%', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('churn_by_tenure_group.png', bbox_inches='tight')
plt.show()

## 5. Feature Engineering and Data Preparation

In [ ]:
# Prepare modeling dataset
df_model = df.drop('customerID', axis=1).copy()

# Encode target variable
df_model['Churn'] = (df_model['Churn'] == 'Yes').astype(int)

# Label encode all categorical columns
le = LabelEncoder()
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print('Categorical columns encoded:', cat_cols)

for col in cat_cols:
    df_model[col] = le.fit_transform(df_model[col])

print('\nFinal encoded dataset shape:', df_model.shape)
df_model.head()

In [ ]:
# Train-test split
X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print('Training set size:', X_train.shape)
print('Test set size:', X_test.shape)
print('Churn rate in train:', round(y_train.mean() * 100, 2), '%')
print('Churn rate in test:', round(y_test.mean() * 100, 2), '%')

In [ ]:
# Feature scaling for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print('Feature scaling applied.')

## 6. Model Training

In [ ]:
# Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train_scaled, y_train)

y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

print('Logistic Regression trained successfully.')

In [ ]:
# Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

print('Random Forest trained successfully.')

## 7. Model Evaluation

In [ ]:
def evaluate_model(name, y_true, y_pred, y_prob):
    return {
        'Model': name,
        'Accuracy (%)': round(accuracy_score(y_true, y_pred) * 100, 2),
        'Precision (%)': round(precision_score(y_true, y_pred) * 100, 2),
        'Recall (%)': round(recall_score(y_true, y_pred) * 100, 2),
        'F1-Score (%)': round(f1_score(y_true, y_pred) * 100, 2),
        'ROC-AUC (%)': round(roc_auc_score(y_true, y_prob) * 100, 2)
    }

results_lr = evaluate_model('Logistic Regression', y_test, y_pred_lr, y_prob_lr)
results_rf = evaluate_model('Random Forest', y_test, y_pred_rf, y_prob_rf)

results_df = pd.DataFrame([results_lr, results_rf])
results_df.set_index('Model', inplace=True)
print('Model Comparison:')
results_df

In [ ]:
# Detailed classification reports
print('=== Logistic Regression Classification Report ===')
print(classification_report(y_test, y_pred_lr, target_names=['No Churn', 'Churn']))

print('=== Random Forest Classification Report ===')
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, title in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Logistic Regression', 'Random Forest']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['No Churn', 'Churn'],
                yticklabels=['No Churn', 'Churn'])
    ax.set_title(f'{title} - Confusion Matrix', fontsize=12, fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(7, 5))

for y_prob, label, color in [
    (y_prob_lr, f'Logistic Regression (AUC = {results_lr["ROC-AUC (%)"]:.2f}%)', '#3b82d4'),
    (y_prob_rf, f'Random Forest (AUC = {results_rf["ROC-AUC (%)"]:.2f}%)', '#7c5cd8')
]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    ax.plot(fpr, tpr, label=label, linewidth=2, color=color)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')
ax.set_title('ROC Curves - Model Comparison', fontsize=13, fontweight='bold')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Metrics bar chart comparison
metrics = ['Accuracy (%)', 'Precision (%)', 'Recall (%)', 'F1-Score (%)', 'ROC-AUC (%)']
x = np.arange(len(metrics))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars1 = ax.bar(x - width/2, [results_lr[m] for m in metrics], width,
               label='Logistic Regression', color='#3b82d4')
bars2 = ax.bar(x + width/2, [results_rf[m] for m in metrics], width,
               label='Random Forest', color='#7c5cd8')

ax.set_title('Model Performance Comparison', fontsize=13, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_ylabel('Score (%)')
ax.set_ylim(0, 100)
ax.legend()

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('model_comparison.png', bbox_inches='tight')
plt.show()

In [ ]:
# Random Forest Feature Importance
feat_importance = pd.Series(
    rf_model.feature_importances_, index=X.columns
).sort_values(ascending=False).head(10)

print('Top 10 Feature Importances (Random Forest):')
print(feat_importance.round(4))

fig, ax = plt.subplots(figsize=(8, 5))
feat_importance.sort_values().plot(kind='barh', ax=ax, color='#3b82d4')
ax.set_title('Top 10 Feature Importances (Random Forest)', fontsize=13, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', bbox_inches='tight')
plt.show()

## 8. Model Selection

In [ ]:
print('=== Model Comparison Summary ===')
print(results_df.to_string())
print()
print('Selected Model: Logistic Regression')
print('Reason: Logistic Regression achieves higher Accuracy (79.39%), Recall (56.42%),'
      ' F1-Score (59.27%), and ROC-AUC (83.45%) compared to Random Forest.')
print('In churn prediction, higher Recall is critical to catch more at-risk customers.')

## 9. Customer Churn Predictions

In [ ]:
# Generate predictions using best model (Logistic Regression)
# Include Customer ID for traceability
customer_ids_test = df.loc[X_test.index, 'customerID'].values

predictions_df = pd.DataFrame({
    'Customer ID':         customer_ids_test,
    'Actual_Churn':        y_test.values,
    'Predicted_Churn':     y_pred_lr,
    'Churn_Probability':   y_prob_lr.round(4),
})
predictions_df['Risk Level'] = predictions_df['Churn_Probability'].apply(
    lambda x: 'High' if x > 0.7 else ('Medium' if x > 0.4 else 'Low')
)

print('Predictions on Test Set (first 10 rows):')
predictions_df.head(10)

In [ ]:
# High-risk customers (churn probability > 0.7)
high_risk = predictions_df[predictions_df['Churn_Probability'] > 0.7]
print(f'High-risk customers (probability > 70%): {len(high_risk)}')
print(f'Actual churners among high-risk: {high_risk["Actual_Churn"].sum()}')
if len(high_risk) > 0:
    print(f'Precision on high-risk group: {round(high_risk["Actual_Churn"].mean()*100, 2)}%')
print()
print('Risk Level Distribution:')
print(predictions_df['Risk Level'].value_counts())

## 10. Key Findings and Business Recommendations

In [ ]:
print('===== KEY FINDINGS =====')
print()
print('1. Overall Churn Rate: 26.58% of customers churned.')
print()
print('2. Contract Type Impact:')
print('   - Month-to-month: 42.71% churn rate (highest)')
print('   - One year:       11.28% churn rate')
print('   - Two year:        2.85% churn rate (lowest)')
print()
print('3. Internet Service Impact:')
print('   - Fiber optic: 41.89% churn rate')
print('   - DSL:         19.00% churn rate')
print('   - No service:   7.43% churn rate')
print()
print('4. Payment Method Impact:')
print('   - Electronic check: 45.29% churn (highest)')
print('   - Mailed check:     19.20% churn')
print('   - Bank transfer:    16.73% churn')
print('   - Credit card:      15.25% churn (lowest among payment methods)')
print()
print('5. Tenure: Churned customers averaged 17.98 months vs 37.65 months for retained customers.')
print('   Customers with tenure < 12 months churn at 47.68% — the highest-risk tenure group.')
print()
print('6. Monthly Charges: Churned customers paid avg $74.44/month vs $61.31 for retained.')
print()
print('===== BUSINESS RECOMMENDATIONS =====')
print()
print('1. Promote long-term contracts: Incentivize month-to-month customers to switch to')
print('   annual or two-year contracts with discounts or loyalty benefits.')
print()
print('2. Improve Fiber Optic service quality: Investigate and address the high churn rate')
print('   among Fiber optic subscribers — likely related to pricing or service reliability.')
print()
print('3. Target electronic check users: This group churns at 45.29%. Offer incentives')
print('   to switch to automatic payment methods, which show lower churn rates.')
print()
print('4. Early retention efforts: Customers with tenure < 12 months churn at 47.68%.')
print('   Onboarding programs and proactive check-ins in the first year can improve retention.')
print()
print('5. Deploy the churn prediction model: Use Logistic Regression (ROC-AUC 83.45%)')
print('   to proactively identify at-risk customers and apply targeted interventions.')

---
**End of Notebook**

| Metric | Logistic Regression | Random Forest |
|---|---|---|
| Accuracy | 79.39% | 78.32% |
| Precision | 62.43% | 61.77% |
| Recall | 56.42% | 48.40% |
| F1-Score | 59.27% | 54.27% |
| ROC-AUC | 83.45% | 81.13% |

**Best Model: Logistic Regression** — outperforms Random Forest across all metrics.

## 11. Conclusion

This project built a customer churn prediction system for a telecom company using the IBM Telco Customer Churn dataset (7,032 records, 20 features).

**Data Cleaning:** TotalCharges had 11 blank entries that were converted and dropped. No duplicate rows were found.

**EDA Key Facts:**
- Overall churn rate: 26.58%
- Month-to-month contract: 42.71% churn vs 2.85% for two-year
- Fiber optic internet: 41.89% churn (highest service type)
- Electronic check payment: 45.29% churn (highest payment method)
- Customers with tenure < 12 months: 47.68% churn (highest-risk group)

**Model Results:**

| Metric | Logistic Regression | Random Forest |
|---|---|---|
| Accuracy | 79.39% | 78.32% |
| Precision | 62.43% | 61.77% |
| Recall | 56.42% | 48.40% |
| F1-Score | 59.27% | 54.27% |
| ROC-AUC | 83.45% | 81.13% |

**Selected Model: Logistic Regression** — achieves higher Recall (56.42%) and ROC-AUC (83.45%), making it more effective at identifying churners early.

**Top Recommendations:** Promote long-term contracts, address fiber optic quality issues, encourage automatic payment methods, implement early-tenure onboarding programs, and deploy the churn model for proactive customer retention.